In [ ]:
import pandas as pd
import numpy as np

import xbbg
from xbbg import blp, ext

# These examples use pandas operations; xbbg otherwise defaults to native Arrow.
xbbg.set_backend('pandas')

# xbbg Examples

This notebook demonstrates reference, historical, intraday, and streaming requests with the current xbbg API. Live requests require an authorized Bloomberg connection and the relevant data entitlements.

The examples select the pandas backend explicitly. Reference and historical results are long-form by default; use `format='semi_long'` when fields should be columns. Bulk results preserve Bloomberg's subfield labels, and intraday bars expose `ticker` and `time` columns rather than an index.

## Reference Data (`BDP` and `BDS`)

Single point-in-time reference data and bulk/block data retrieval.

In [ ]:
xbbg.__version__

In [ ]:
blp.bdp('AAPL US Equity', flds=['Security_Name', 'Last_Price'])

In [ ]:
blp.bdp('6758 JP Equity', flds='Crncy_Adj_Mkt_Cap', Eqy_Fund_Crncy='USD')

In [ ]:
holders = blp.bds('AMZN US Equity', flds='All_Holders_Public_Filings')
holders.head()

In [ ]:
ext.dividend('SPY US Equity', start_date='2019-01-01')

In [ ]:
ext.earnings('META US Equity', year=2018, periods=2)

## Historical Data (`BDH`)

End-of-day historical data with Excel-compatible inputs and dividend/split adjustments.

### Excel-Compatible Overrides

Use familiar Excel-style parameters like `Per` (periodicity), `Fill` (fill method), and `Days` (non-trading day fill option).

In [ ]:
blp.bdh(
    tickers='SHCOMP Index', flds=['high', 'low', 'last_price'],
    start_date='2019-11-01', end_date='2020-01-01', Per='W', Fill='P', Days='A',
)

### Dividend and Split Adjustments

Compare adjusted vs. unadjusted prices using `CshAdjNormal`, `CshAdjAbnormal`, and `CapChg` parameters.

In [ ]:
pd.concat([
    blp.bdh(
        'AAPL US Equity', 'PX_LAST', '2014-06-05', '2014-06-10',
        format='semi_long', CshAdjNormal=True, CshAdjAbnormal=True, CapChg=True,
    ).set_index(['ticker', 'date'])[['PX_LAST']].rename(columns={'PX_LAST': 'Px_Adj'}),
    blp.bdh(
        'AAPL US Equity', 'PX_LAST', '2014-06-05', '2014-06-10',
        format='semi_long', CshAdjNormal=False, CshAdjAbnormal=False, CapChg=False,
    ).set_index(['ticker', 'date'])[['PX_LAST']].rename(columns={'PX_LAST': 'Px_Raw'}),
], axis=1)

## Intraday Bars (`BDIB`)

Retrieve intraday bar data with minute or second-level granularity, session filtering, and exchange-aware market hours. The previous-weekday calculation below skips weekends, not exchange holidays; choose another trading date if a market was closed.

In [ ]:
pre_dt = (pd.Timestamp.now(tz='America/New_York') - pd.offsets.BDay()).date()
pre_dt

In [ ]:
blp.bdib('QQQ US Equity', dt=pre_dt, session='day', request_tz='exchange').tail()

In [ ]:
blp.bdib('388 HK Equity', dt=pre_dt, session='am_open_7', request_tz='exchange')

## Intraday Tick Data (`BDTICK`)

Tick-by-tick data with event types, condition codes, and exchange information.

In [ ]:
blp.bdtick(
    'QQQ US Equity',
    start_datetime=f'{pre_dt.isoformat()} 09:30:00',
    end_datetime=f'{pre_dt.isoformat()} 09:35:00',
    event_types=['TRADE'],
    request_tz='America/New_York',
    output_tz='America/New_York',
).tail(10)

## Equity Screening (`BEQS`)

Bloomberg Equity Screening allows you to query stocks based on custom criteria and predefined screens.

In [ ]:
blp.beqs('Core Capital Ratios', screen_type='GLOBAL').iloc[:5, :6]

## Real-time Subscriptions

Stream live market data updates using async subscriptions.

### Live Data Streaming

`blp.asubscribe(..., output='dict')` yields sparse dictionary updates. Missing fields are unchanged; fields present with `None` clear the previous value. The async context manager unsubscribes on exit. This example waits for one update, so run it when the security is active or interrupt the cell to stop waiting.

In [ ]:
async with await blp.asubscribe(
    'AAPL US Equity', ['LAST_PRICE', 'BID', 'ASK'], output='dict',
) as subscription:
    async for update in subscription:
        print(update)
        break

## Data Pipelines

Use pandas to reshape the explicit `time`, `close`, and `volume` columns into ticker/field columns for the calculations below.

In [ ]:
pre_dt = (pd.Timestamp.now(tz='Asia/Tokyo') - pd.offsets.BDay()).date()

fx = (
    blp.bdib(
        'JPY Curncy', dt=pre_dt,
        request_tz='Asia/Tokyo', output_tz='Asia/Tokyo',
    )
    .set_index('time')['close']
    .sort_index()
)
jp = pd.concat({
    ticker: blp.bdib(
        ticker, dt=pre_dt, session='day',
        request_tz='exchange', output_tz='Asia/Tokyo',
    ).set_index('time')[['close', 'volume']]
    for ticker in ['7974 JP Equity', '9984 JP Equity']
}, axis=1).sort_index()
jp.tail()

### Currency Conversion

`JPY Curncy` is quoted in JPY per USD. Divide the JPY closing prices by the most recent FX bar at or before each equity bar; do not backfill from future quotes.

In [ ]:
prices = jp.xs('close', axis=1, level=1)
prices_usd = prices.div(fx.reindex(prices.index, method='ffill'), axis=0)
prices_usd.tail()

### Custom Pipelines

Create reusable pandas functions and apply them with `DataFrame.pipe`.

#### Volume-Weighted Average Price (VWAP)

Estimate VWAP using each bar's closing price and volume. This is a bar-close approximation, not tick-level VWAP; FX-adjusted calculations use the contemporaneous FX bar.

In [ ]:
def vwap(data: pd.DataFrame, fx=None, name=None) -> pd.Series:
    closes = data.xs('close', axis=1, level=1)
    volumes = data.xs('volume', axis=1, level=1)
    if fx is not None:
        closes = closes.div(fx.reindex(data.index, method='ffill'), axis=0)
    weights = volumes.where(closes.notna())
    return (
        closes.mul(weights).sum()
        .div(weights.sum().replace(0, np.nan))
        .rename(name)
    )

#### VWAP in Local Currency

In [ ]:
jp.pipe(vwap, name=jp.index[-1].date())

#### VWAP in USD

Convert prices to USD before calculating VWAP using FX rates.

In [ ]:
jp.pipe(vwap, fx=fx, name=jp.index[-1].date())

#### Time-of-Day Volume Analysis

Calculate cumulative traded volume up to a specific time of day across multiple trading days.

In [ ]:
jp_hist = pd.concat([
    pd.concat({
        ticker: blp.bdib(
            ticker, dt=dt.date(), session='day',
            request_tz='exchange', output_tz='Asia/Tokyo',
        ).set_index('time')[['close', 'volume']]
        for ticker in ['7974 JP Equity', '9984 JP Equity']
    }, axis=1)
    for dt in pd.bdate_range(end=pre_dt, periods=9)
], sort=False).sort_index()
np.unique(jp_hist.index.date)

In [ ]:
def drop_zeros(data: pd.DataFrame) -> pd.DataFrame:
    return (
        data
        .replace(0, np.nan)
        .dropna(how='all')
        .replace(np.nan, 0)
    )

In [ ]:
def traded_volume(data: pd.DataFrame, asof: str) -> pd.DataFrame:
    return (
        data.xs('volume', axis=1, level=1)
        .between_time('00:00', asof)
        .resample('B')
        .sum()
        .pipe(drop_zeros)
    )

In [ ]:
jp_hist.pipe(traded_volume, asof='10:00')

In [ ]:
jp_hist.pipe(traded_volume, asof='11:00')